In [1]:
from value import Value
import random

In [2]:
class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []

class Neuron(Module):

    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0)

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer(Module):

    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

class MLP(Module):

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], nonlin=i!=len(nouts)-1) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

In [3]:
x = [2.0, 3.0, -1.0]
n = MLP(3,[4,4,1])

n(x)

Value(data = 0.6722968623389801)

In [4]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0]

In [5]:
for k in range(20):

    #forward pass
    ypred = [n(x) for x in xs]
    loss = sum([(ygt - yout)**2 for ygt, yout in zip(ys,ypred)])

    # backward pass
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    for p in n.parameters():
        p.data += -0.01 * p.grad

    print(k, loss.data)


0 1.6851207270804276
1 1.2816296090407562
2 1.0867403743339672
3 0.9646467364473295
4 0.874756175396349
5 0.8023877898527386
6 0.7411973372071428
7 0.6880050080349185
8 0.6409942534989556
9 0.5990037937946351
10 0.5612213681693092
11 0.5270394385851056
12 0.4959813908342037
13 0.46766065537975926
14 0.44175628122572397
15 0.4179973009377631
16 0.3961521078436149
17 0.37602087149009933
18 0.357429900779035
19 0.34022731889955193


In [6]:
ypred

[Value(data = 0.7227919446659534),
 Value(data = -0.8466297026706154),
 Value(data = -0.5805654219512225),
 Value(data = 0.7471458926740396)]